# IDS basado en IA — Huella digital IEEE 802.3 · Cierre (E7)

**Objetivos del artículo**
- **OE1**: caracterizar vectores de ataque por capa TCP/IP e ISO/OSI.
- **OE2**: módulo de huella digital en tiempo real + RF/XGBoost sobre un dataset de referencia.
- **OE3**: evaluar el desempeño simulando ataque a la huella; diagnóstico, mitigación y pronóstico.

Este cuaderno recorre el proyecto de punta a punta (CRISP-DM) y reproduce las cifras que ya están
congeladas en `reports/`. Ejecuta las celdas en orden desde la raíz del repo.

```
Captura (Scapy) -> Huella IEEE 802.3 (src/ventanas.py) -> Preprocesado
   -> Clasificacion (RF/XGBoost)  +  Identidad (src/identidad.py)
   -> Mitigacion (dry-run)  +  Pronostico
                     ^
          Evasion a nivel de pcap (E4) / streaming en vivo (E5)
```

In [ ]:
import json, sys
from pathlib import Path
RAIZ = Path.cwd()
assert (RAIZ / "src").exists(), "Ejecuta el cuaderno desde la raiz del repositorio."
if str(RAIZ) not in sys.path: sys.path.insert(0, str(RAIZ))
import numpy as np, pandas as pd

def reporte(nombre):
    p = RAIZ / "reports" / nombre
    return json.loads(p.read_text()) if p.exists() else None

print("Reportes:", sorted(p.name for p in (RAIZ/'reports').glob('*.json'))[:20])

## OE1 - Vectores de ataque por capa

Para cada ataque, el invariante de efecto (la fisica del objetivo) fija la tasa minima que lo
mantiene siendo un ataque; es lo que E4 usa para separar evasion de 'el ataque desaparecio'.

In [ ]:
inv = reporte("robustez_e4c.json")
filas = inv["invariantes"] if inv else []
cat = {"mac_flooding":"enlace","arp_spoofing":"enlace","icmp_flood":"red",
       "syn_flood":"transporte","port_scan":"transporte","dns_spoofing":"aplicacion"}
pd.DataFrame([{ "ataque":x["ataque"], "capa":cat.get(x["ataque"],"?"),
               "invariante":x["invariante"], "suelo":x["suelo"]} for x in filas])

## OE2 - Clasificador supervisado (E2)

RF y XGBoost sobre las 52 columnas, 5 folds por sesion x 5 semillas, umbral a 1 FA/disp-dia.

In [ ]:
e2 = reporte("evaluacion_e2.json")
if e2:
    for m, d in e2["modelos"].items():
        r = d["resumen"]
        print(f"{m:14s} recall/episodio {r['episodios']['recall']['valor']:.3f}"
              f"  FA/disp-dia {r['dispositivos']['falsas_alarmas_por_dispositivo_dia']['valor']:.2f}"
              f"  PR-AUC {r['ventanas_referencia']['pr_auc']['valor']:.3f}")
else:
    print("Genera con: .venv/bin/python scripts/evaluar.py --config configs/e2.json")

## OE2/OE3 - Identidad del dispositivo (E3)

Perfil de pila por equipo + vinculo IP->MAC. Deteccion por nivel de clon.

In [ ]:
e3 = reporte("identidad_e3b.json")
if e3 and "niveles" in e3:
    t = e3["niveles"]["tabla"]
    display(pd.DataFrame([{ "nivel":n, "deteccion":round(v["deteccion"]["valor"],3),
                            "latencia_s":round((v["latencia_s_mediana"]["valor"] or 0),1)} for n,v in t.items()]))
print("IP-ID por flujo (E3B): FA 16,19 -> 0,38 /disp-dia")

## OE3 - Evasion a nivel de pcap (E4)

Los floods pierden efecto antes de evadir el etiquetado; el ARP lento es invisible al supervisado
pero lo caza el vigilante IP->MAC.

In [ ]:
r4 = reporte("robustez_e4.json"); r4c = reporte("robustez_e4c.json")
if r4: print("E4R recall/intensidad:", r4["resumen"]["recall_por_intensidad"], "| invisibles:", r4["resumen"]["invisibles_total"])
if r4c:
    arp = r4c["resumen"]["recall_arp_por_paso"]
    print("E4C ARP supervisado por paso:", {k:v["supervisado"] for k,v in arp.items()})
    print("E4C ARP vigilante @5s:", arp["5s"]["vigilante"], "| FA vigilante:", r4c["resumen"]["fa_vigilante_variacion_por_dispositivo_dia"])

## OE3 - Captura en streaming (E5)

Demo offline: reproducimos una sesion y mostramos las alertas de identidad (sin root).

In [ ]:
from src.streaming import MotorStreaming
from src.ventanas import leer_pcap
pcap = RAIZ/"data"/"pcap_clones"/"clones_000.pcap"
if pcap.exists():
    motor = MotorStreaming(modelo=None, con_identidad=True, red_local="10.20.0.0/16")
    res = motor.reproducir(leer_pcap(pcap))
    print("ventanas:", motor.metricas["ventanas"], "| alarmas identidad:", sum(r.alarma_identidad for r in res))
    print("metricas:", motor.resumen_metricas())
else:
    print("Regenera data/pcap_clones con los generadores congelados para la demo.")

### Menos falsas alarmas sin absorber el clon (E5C)

La tercera via de aprendizaje baja las FA con variacion benigna de 826 a 382 /disp-dia.

In [ ]:
a = reporte("adopcion_e5c.json")
if a: print("FA con variacion:", a["resumen"]["fa_con_variacion"], "| deteccion ritmo:", a["resumen"]["deteccion_por_nivel"]["ritmo"])

## OE3 - Mitigacion y pronostico (E6)

Mitigacion escalable en dry-run (imprime el comando, no toca la red) y pronostico de la tasa de
alertas para anticipar un escalamiento.

In [ ]:
from src.mitigacion import GestorMitigacion, PoliticaMitigacion
g = GestorMitigacion(politica=PoliticaMitigacion(umbral_bloqueo=3, ventana_s=60))
for i in range(4):
    nivel = g.procesar_alerta("02:ff:00:00:00:99", ts_s=float(i), score=0.9)
print("modo:", g.modo, "| ultimo nivel:", nivel.name, "| acciones:", [x["accion"] for x in g.acciones])

In [ ]:
from src.pronostico import PronosticadorAnomalias
serie = np.array([0,0,1,1,2,3,4,6,8], dtype=float)
p = PronosticadorAnomalias(metodo="holt").entrenar(serie)
print("tendencia:", p.tendencia, "| pronostico 3 pasos:", p.pronosticar(3).round(2).tolist())
print("aviso temprano:", p.aviso_temprano(umbral=10))

## Conclusiones

1. El supervisado detecta el catalogo conocido (recall/episodio ~0,894) pero no transfiere a tipos
   nuevos (LOAO ~0,228).
2. Un detector solo-benigno no lo arregla a este presupuesto de FA (~0,185).
3. El perfil de pila detecta clones de pila distinta en segundos, a 0,38 FA/disp-dia.
4. El sistema se rompe en el clon de pila completa (~0,20, solo con contador de IP-ID global; 0 de
   47 con Linux/macOS/Android).
5. Con variacion benigna las FA suben a 826/disp-dia; la tercera via (E5C) las baja a 382 sin perder
   deteccion, salvo dos_pilas (ambiguedad de fondo).
6. Evasion a nivel de pcap: los floods pierden efecto antes de evadir; el ARP lento es invisible al
   supervisado (1,000->0,000 a paso >= 5 s) pero el vigilante IP->MAC lo caza al 1,000 con 0 FA.
7. El camino en vivo (E5) reproduce el lote con paridad; la mitigacion (E6) actua en dry-run y el
   pronostico anticipa el escalamiento.

Limites para el laboratorio real: FA del vigilante con DHCP/failover; imitacion de huella a nivel de
paquete; RST de la victima frente al clon de pila completa; ejecucion real de la mitigacion; y todo
medido en sesiones de 15 min.